In [ ]:
from mnist import Net, EuclideanNet, train, test
from hyptorch.pmath import dist0

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import cm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import os
import json
import dataclasses as dc

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")

np.random.seed(0)
torch.manual_seed(0);

In [ ]:
job_id = "435797"

dims = [2, 8, 16, 32]
args_paths = []
state_dict_paths = []
for dim in dims:
    args_path = f"checkpoints/{job_id}/mnist_args_{dim}d.json"
    args_paths.append(args_path)
    with open(args_path, "r") as f:
        args = json.load(f)
        state_dict_path = f"checkpoints/{job_id}/mnist_hyp_cnn_{args['best_epoch_hyp']}_{dim}d.pt"
        state_dict_paths.append(state_dict_path)

    assert os.path.isfile(state_dict_path)
    assert os.path.isfile(args_path)

data_path = "../mnist_data"
if os.getenv("GLOBALSCRATCH") is not None:
    data_path = os.path.join(os.getenv("GLOBALSCRATCH"), "mnist_data")

omni_path = os.path.join(data_path, "omniglot")

assert os.path.isdir(data_path)
assert os.path.isdir(omni_path)

In [ ]:
models_hyp = {}

for dim, args_path, state_dict_path in zip(dims, args_paths, state_dict_paths):
    with open(args_path, "r") as f:
        args = json.load(f)

    args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)
    models_hyp[dim] = Net(args).to(device)

    state_dict = torch.load(state_dict_path, map_location=device)
    models_hyp[dim].load_state_dict(state_dict)

In [ ]:
args_paths = []
state_dict_paths = []
for dim in dims:
    args_path = f"checkpoints/{job_id}/mnist_args_{dim}d.json"
    args_paths.append(args_path)
    with open(args_path, "r") as f:
        args = json.load(f)
        state_dict_path = f"checkpoints/{job_id}/mnist_euc_cnn_{args['best_epoch_euc']}_{dim}d.pt"
        state_dict_paths.append(state_dict_path)

    assert os.path.isfile(state_dict_path)
    assert os.path.isfile(args_path)

In [ ]:
models_euc = {}

for dim, args_path, state_dict_path in zip(dims, args_paths, state_dict_paths):
    with open(args_path, "r") as f:
        args = json.load(f)

    args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)
    models_euc[dim] = EuclideanNet(args).to(device)

    state_dict = torch.load(state_dict_path, map_location=device)
    models_euc[dim].load_state_dict(state_dict)

In [ ]:
save_imgs = True
if save_imgs:
    os.makedirs("exp_images", exist_ok=True)

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True} if device.type == "cuda" else {}
train_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.batch_size,
    shuffle=True,
    **kwargs
)

test_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=False,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
# Trained until 95% accuracy
test(None, models_euc[2], device, test_MNIST_loader);

In [ ]:
test_Omniglot_loader = torch.utils.data.DataLoader(
    datasets.Omniglot(
        omni_path,
        background=False,
        download=True,
        transform=transforms.Compose(
            [
                transforms.Resize((28, 28)), 
                transforms.ToTensor(),
                transforms.Lambda(lambda image: 1.0 - image),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
omni_data, omni_target = next(iter(test_Omniglot_loader))
omni_data, omni_target = omni_data.to(device), omni_target.to(device)

num_samples = 20
omni_sample_data = omni_data.detach().cpu().numpy()[:num_samples]
omni_sample_target = omni_target.detach().cpu().numpy()[:num_samples]
omni_sample_embeddings = models_hyp[2].embed(omni_data)[:num_samples]

In [ ]:
data, target = next(iter(test_MNIST_loader))
data, target = data.to(device), target.to(device)
output = models_hyp[2](data)

num_samples = 40
sample_data = data.detach().cpu().numpy()[:num_samples]
sample_target = target.detach().cpu().numpy()[:num_samples]
sample_embeddings = models_hyp[2].embed(data)[:num_samples]

In [ ]:
fig, ax = plt.subplots(10//2, 2, figsize=(6, 8))

for i in range(10):
    ax[i//2, i%2].imshow(sample_data[i, 0], aspect="equal", cmap="gray")
    ax[i//2, i%2].axis("off")

vals = np.exp(models_hyp[2](data).detach().cpu().numpy()[:10])
sample_pred = np.argmax(vals, axis=1)
sample_conf = np.max(vals, axis=1)
for i in range(10):
    ax[i//2, i%2].set_title(f"Pred: {sample_pred[i]}\nConf: {sample_conf[i]:.2f}")

plt.tight_layout()
plt.show()

### Figure 1 from the paper

In [ ]:
np_sample_embeddings = sample_embeddings.detach().cpu().numpy()
np_omni_sample_embeddings = omni_sample_embeddings.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))

circle = plt.Circle((0, 0), 1, color="black", fill=False)
ax.add_artist(circle)

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (x, y) in enumerate(np_sample_embeddings):
    img = sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color=colors[sample_target[i]], s=50)

for i, (x, y) in enumerate(np_omni_sample_embeddings):
    img = omni_sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color="black", s=50, marker="x")

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")

plt.axis("off")
if save_imgs:
    plt.savefig("exp_images/embedding_plot.png")
else:
    plt.show()

### Compare distributions

In [ ]:
def collect_dist0(model, loader, device):
    model.eval()
    distances = []

    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            embedding = model.embed(data)
            dist = dist0(embedding, c=model.tp.c).view(-1)
            distances.extend(dist.cpu().numpy())

    return np.array(distances)

In [ ]:
MNIST_dists = []
omni_dists = []

for dim in dims:
    MNIST_dists.append(collect_dist0(models_hyp[dim], test_MNIST_loader, device))
    omni_dists.append(collect_dist0(models_hyp[dim], test_Omniglot_loader, device))

In [ ]:
fig, axs = plt.subplots(nrows=int(np.ceil(len(dims)/2)), ncols=2, figsize=(10, 6)) 

for i, (dim, MNIST_dist, omni_dist) in enumerate(zip(dims, MNIST_dists, omni_dists)):
    ax = axs[i // 2, i % 2]
    ax.hist(MNIST_dist, bins=50, alpha=0.4, color="red", density=True, label="MNIST")
    ax.hist(omni_dist, bins=50, alpha=0.4, color="blue", density=True, label="Omniglot")
    ax.set_xlabel("Distance to 0")
    ax.set_ylabel("Density")
    ax.set_title(f"Histogram of Distances to 0 (dim={dim})")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
if save_imgs:
    plt.savefig("exp_images/mnist_omni_hist_hyp.png")
else:
    plt.show()

In [ ]:
from scipy.stats import ks_2samp

for i, (dim, MNIST_dist, omni_dist) in enumerate(zip(dims, MNIST_dists, omni_dists)):
    ks_stat, ks_p_value = ks_2samp(MNIST_dist, omni_dist)
    print(f"KS Statistic: {ks_stat}\nKS p-value: {ks_p_value}")

In [ ]:
def euc_pmax(model, loader, device):
    model.eval()
    pmaxes = []

    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            pmax = model(data).exp().max(dim=1)[0]
            pmaxes.extend(pmax.cpu().numpy())

    return np.array(pmaxes)


In [ ]:
MNIST_pmaxs = []
omni_pmaxs = []

for dim in dims:
    MNIST_pmaxs.append(euc_pmax(models_euc[dim], test_MNIST_loader, device))
    omni_pmaxs.append(euc_pmax(models_euc[dim], test_Omniglot_loader, device))

In [ ]:
for i, (dim, MNIST_pmax, omni_pmax) in enumerate(zip(dims, MNIST_pmaxs, omni_pmaxs)):
    ks_stat, ks_p_value = ks_2samp(MNIST_pmax, omni_pmax)
    print(f"KS Statistic: {ks_stat}\nKS p-value: {ks_p_value}")

The Euclidean model systematically obtains lower Kolmogorov-Smirnov score compared to the Hyperbolic.